# Writing summary

This default notebook is executed using a Lakeflow job as defined in resources/sample_job.job.yml.

In [0]:
# Set default catalog and schema
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{schema}`")
spark.sql(f"USE SCHEMA `{schema}`")

from pyspark.sql import functions as F

In [0]:
df = spark.read.table("samples.nyctaxi.trips")


In [0]:

daily_stats = (
    df.withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
    .groupBy("pickup_date")
    .agg(
        F.avg("trip_distance").alias("avg_trip_distance"),
        F.sum("fare_amount").alias("total_fare_amount"),
    )
    .orderBy("pickup_date")
)

table_name = f"`{catalog}`.`{schema}`.daily_stats"

# # Check if the table exists before writing - if not, do not proceed
# if spark.catalog.tableExists(table_name):
#     raise ValueError(f"Table {table_name} exists - aborting write.")

# writing to the catalog
(
    daily_stats.write.format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)


In [0]:
display(
    spark.read.table(f"`{catalog}`.`{schema}`.daily_stats")
    .groupBy(F.year(F.col('pickup_date')).alias('year'), F.month(F.col('pickup_date')).alias('month'))
    .agg(
        F.round(F.avg("avg_trip_distance"), 2).alias("avg_trip_distance"),
        F.sum("total_fare_amount").alias("total_fare_amount"),
    )
)